# 3. La prédiction par cohorte

La même régression que dans `2_Prediction`, sur le même tableau (réglages de `reglages.py`) : les mêmes six modèles, un train (80 %), un test (20 %), les mêmes métriques à la fin.

La seule différence : au lieu d'un seul modèle pour toute la banque, **chaque cohorte a son modèle**.

Pour une cohorte, ça se passe comme ça :

1. on prend les salariés de la cohorte ;
2. ceux qui sont dans le train servent à apprendre le modèle ;
3. le modèle prédit ceux qui sont dans le test.

On fait pareil pour toutes les cohortes, on rassemble les prédictions, et on calcule les métriques.

Il y a plusieurs façons de former les cohortes (avec l'ancienneté, le diplôme…). Le notebook les essaie toutes, donne les métriques de chacune, puis les classe de la meilleure à la moins bonne.

La ligne **Sans cohorte** des tableaux est le repère : on ne découpe rien, un seul modèle apprend sur tout le train.

---
## 1. Le tableau et les colonnes

Le même tableau que dans `1_Visualisations` et `2_Prediction` : la détection tourne, les anomalies de `A_RETIRER` sortent, puis les valeurs aberrantes si `ENLEVER = True`.

Les colonnes que les modèles reçoivent sont celles de la liste `COLONNES`. Une colonne qui a servi à former la cohorte est la même pour tous dans la cohorte : elle n'explique plus rien à l'intérieur. Elle reste dans la liste pour le modèle « Sans cohorte » et pour la cohorte « Autres ».

In [ ]:
import importlib

import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split

# Les trois fichiers communs du dossier : les réglages, la préparation du tableau, les modèles.
# importlib.reload les relit : un réglage changé dans reglages.py est pris en compte dès qu'on relance cette cellule.
import reglages, preparation, modeles
for fichier in (reglages, preparation, modeles):
    importlib.reload(fichier)
from preparation import CIBLES, RESULTATS, choisir_colonnes, outliers, preparer_tableau
from modeles import creer_modeles, metriques, salaire_predit

px.defaults.template = 'plotly_white'   # des graphes sur fond blanc
pd.set_option('display.precision', 2)   # les tableaux affichent 2 chiffres après la virgule ; les calculs gardent toute leur précision
pd.set_option('display.max_colwidth', None)   # les textes des tableaux s'affichent en entier

tableau = preparer_tableau()
df = tableau.df   # les salariés gardés

# Les deux cibles et les colonnes de la liste COLONNES, en chiffres
donnees = choisir_colonnes(df)
# Les colonnes que les modèles reçoivent : celles de la liste, sans les deux cibles
X = donnees.drop(columns=CIBLES)

---
## 2. Les cohortes à essayer

C'est ici qu'on règle les cohortes. Dans `COHORTES`, **une ligne = une façon de former les cohortes** :

* à gauche, le nom qu'on lui donne (il sert dans les tableaux) ;
* à droite, entre crochets, les colonnes qui forment la cohorte.

Une cohorte, ce sont tous les salariés qui ont les mêmes valeurs dans ces colonnes. Avec `['Niveau', 'Job_Family']`, les salariés de niveau 6 en Retail Banking forment une cohorte, ceux de niveau 5 en Finance une autre. Chaque salarié est dans une seule cohorte.

Une cohorte de moins de `TAILLE_MIN` salariés est trop petite pour apprendre un modèle : ses salariés vont dans une cohorte à part, « Autres ».

Sous la cellule, le tableau donne pour chaque ligne de `COHORTES` : le nombre de cohortes qu'elle forme, et parmi elles combien ont moins de `TAILLE_MIN` salariés.

Dans la liste, les commentaires séparent les cohortes qui existaient déjà (les trois groupes de la détection) de celles ajoutées dans ce notebook.

Pour ajouter une façon de faire : recopier une ligne et changer les colonnes. Un `#` devant une ligne la met de côté.

Les colonnes qu'on peut utiliser :

| Colonne | Ce que c'est |
|---|---|
| `Niveau` | le niveau du poste |
| `Job_Family` | le métier |
| `Anciennete_Bucket` | la tranche d'ancienneté (0-2, 3-5, 6-12, 13-20, plus de 20 ans) |
| `Niveau_Etude_Max` | le niveau d'études (BAC, BAC+2, BAC+3…) |
| `Tranche_Etude` | le niveau d'études en trois tranches (BAC+2 ou moins, BAC+3 ou BAC+4, BAC+5 et plus) |
| `Entite_N1` | le pôle |
| `Performance_N1`, `Positionnement_9BOX`, `Hot_job` | la performance, la case 9-Box, la tension du métier |

Une case vide (une tranche de diplôme inconnue, par exemple) s'écrit « nan » dans le nom de la cohorte : les salariés concernés sont ensemble.

In [ ]:
# La taille minimale d'une cohorte : celle de la détection (15). Mettre un autre nombre ici pour la changer.
TAILLE_MIN = tableau.regles['cohort_min_size']

COHORTES = {
    # ----- LES COHORTES QUI EXISTAIENT DÉJÀ : les trois groupes de la détection
    #       (config/rules.yaml, réglage cohort_widening_steps)
    'Niveau + métier + ancienneté':           ['Niveau', 'Job_Family', 'Anciennete_Bucket'],
    'Niveau + métier':                        ['Niveau', 'Job_Family'],
    'Niveau seul':                            ['Niveau'],

    # ----- LES COHORTES AJOUTÉES DANS CE NOTEBOOK : elles n'existent nulle part ailleurs dans le projet
    #       Avec le niveau d'études :
    'Niveau + métier + diplôme':              ['Niveau', 'Job_Family', 'Niveau_Etude_Max'],
    'Niveau + métier + ancienneté + diplôme': ['Niveau', 'Job_Family', 'Anciennete_Bucket', 'Niveau_Etude_Max'],
    'Niveau + métier + tranche de diplôme':   ['Niveau', 'Job_Family', 'Tranche_Etude'],
    #       Sans le niveau :
    'Métier seul':                            ['Job_Family'],
    #       Pour en ajouter une autre, écrire sa ligne ici, sur le même modèle.
}
AUTRES = 'Autres (cohortes trop petites)'

# Une façon de faire qui cite une colonne absente du tableau est mise de côté, avec un message
for nom in [nom for nom, colonnes in COHORTES.items() if any(c not in df.columns for c in colonnes)]:
    print(f"« {nom} » mise de côté : colonne absente du tableau ({[c for c in COHORTES[nom] if c not in df.columns]}).")
    del COHORTES[nom]

# Pour chaque façon de faire : la cohorte de chaque salarié, et le compte des cohortes pour le tableau
cohorte = pd.DataFrame(index=df.index)
tableau_cohortes = {}
for nom, colonnes in COHORTES.items():
    # Le nom de la cohorte : les valeurs du salarié dans ces colonnes, collées avec des « | » (« 6 | RETAIL BANKING | 0-2 »).
    # fillna('nan') : une case vide s'écrit « nan », quelle que soit la version de pandas.
    nom_cohorte = df[colonnes].astype(str).fillna('nan').agg(' | '.join, axis=1)
    # Une cohorte de moins de TAILLE_MIN salariés est trop petite : ses salariés vont dans « Autres »
    taille = nom_cohorte.value_counts()   # le nombre de salariés de chaque cohorte
    trop_petite = nom_cohorte.map(taille) < TAILLE_MIN
    cohorte[nom] = nom_cohorte.mask(trop_petite, AUTRES)
    # Pour le tableau : combien de cohortes en tout, et combien n'atteignent pas la taille minimale
    tableau_cohortes[' + '.join(colonnes)] = {'Nombre de cohortes': len(taille),
                                              f'Dont moins de {TAILLE_MIN} salariés': int((taille < TAILLE_MIN).sum())}

# Le tableau des cohortes : une ligne par façon de former les cohortes
tableau_cohortes = pd.DataFrame(tableau_cohortes).T.rename_axis('Cohortes')
display(tableau_cohortes)

---
## 3. Le train et le test

On coupe une seule fois, comme dans `2_Prediction` : 80 % des salariés dans le train, pour apprendre, et 20 % dans le test, pour vérifier.

Chaque cohorte a donc des salariés dans le train et des salariés dans le test.

Pourquoi couper une seule fois, et pas cohorte par cohorte : toutes les façons de former les cohortes sont ainsi jugées sur les mêmes salariés du test. On peut les comparer.

In [ ]:
# 1. On coupe en train et test. random_state : le même découpage à chaque lancement.
train, test = train_test_split(X.index, test_size=0.2, random_state=42)

# 2. Une cohorte qui n'a personne dans le train ne peut pas apprendre : ses salariés sortent du test,
#    pour toutes les façons de faire (elles restent jugées sur les mêmes salariés)
sans_train = pd.DataFrame({nom: ~cohorte.loc[test, nom].isin(cohorte.loc[train, nom]) for nom in COHORTES}).any(axis=1)
if sans_train.any():
    test = test[~sans_train.to_numpy()]
    print(f"{int(sans_train.sum())} salariés sortis du test : leur cohorte n'a personne dans le train.")
print(f"Train : {len(train)} salariés. Test : {len(test)} salariés.")

---
## 4. Les six modèles, par cohorte

Les mêmes six modèles que dans `2_Prediction` (fichier `modeles.py`), avec les mêmes réglages : M1, M2 et M5 prédisent le salaire, M3, M4 et M6 le Compa-Ratio ; une régression linéaire, un arbre de décision, un gradient boosting.

Pour chaque façon de former les cohortes : dans chaque cohorte, le modèle apprend sur les salariés de la cohorte qui sont dans le train, et prédit ceux qui sont dans le test.

La dernière ligne des tableaux, **Sans cohorte**, est le repère : on ne découpe rien, un seul modèle apprend une fois sur tout le train.

Cette cellule est la plus longue : elle refait les six modèles pour chaque ligne de `COHORTES`. Une ligne s'affiche à chaque façon de faire terminée. Pour aller plus vite, mettre un `#` devant des lignes de `COHORTES` (les deux gradient boosting, M5 et M6, prennent presque tout le temps).

In [ ]:
# 1. Les six modèles (fichier modeles.py), avec les réglages de reglages.py
MODELES = creer_modeles()
y_test, mid = df.loc[test, 'Fixe_Annuel_MAD'], df.loc[test, 'Mid']


def salaire_predit_par_cohorte(modele, cible, nom):
    # Un modèle par cohorte : il apprend sur les salariés de la cohorte qui sont dans le train,
    # puis prédit les salariés de la cohorte qui sont dans le test
    y_pred = pd.Series(index=test, dtype=float)
    for c in cohorte.loc[test, nom].unique():
        apprend = train[(cohorte.loc[train, nom] == c).to_numpy()]
        verifie = test[(cohorte.loc[test, nom] == c).to_numpy()]
        y_pred.loc[verifie] = salaire_predit(modele, cible, apprend, verifie, X, df)
    return y_pred


# 2. Pour chaque façon de former les cohortes, chaque modèle apprend par cohorte et prédit les salariés du test
REPERE = 'Sans cohorte'
resultats = []
for nom in COHORTES:
    for modele_nom, (quoi, cible, modele) in MODELES.items():
        resultats.append({'Cohorte': nom, 'Modèle': modele_nom, **metriques(cible, y_test, salaire_predit_par_cohorte(modele, cible, nom), mid)})
    print(f"Fait : {nom}")
# Le repère : les mêmes modèles appris une seule fois sur tout le train. Ce sont les chiffres de 2_Prediction.
for modele_nom, (quoi, cible, modele) in MODELES.items():
    resultats.append({'Cohorte': REPERE, 'Modèle': modele_nom, **metriques(cible, y_test, salaire_predit(modele, cible, train, test, X, df), mid)})
resultats = pd.DataFrame(resultats)

# 3. Un tableau par métrique : une ligne par façon de former les cohortes, une colonne par modèle
display(pd.Series({modele_nom: quoi for modele_nom, (quoi, cible, modele) in MODELES.items()}).to_frame('Ce que fait le modèle'))
for metrique in ['MAE (MAD)', 'RMSE (MAD)', 'R²']:
    tableau_metrique = resultats.pivot(index='Cohorte', columns='Modèle', values=metrique).loc[list(COHORTES) + [REPERE]]
    display(tableau_metrique.round(2).style.format('{:.2f}').set_caption(metrique))

# 4. La plus petite MAE de tous les essais
meilleur = resultats.loc[resultats['MAE (MAD)'].idxmin()]
print(f"La plus petite MAE : {meilleur['MAE (MAD)']:.2f} MAD, avec {meilleur['Modèle']} et « {meilleur['Cohorte']} ».")

* **MAE** : l'erreur moyenne, dans l'unité du salaire.
* **RMSE** : pareil, mais les grosses erreurs comptent plus.
* **R²** : sur ce que le modèle prédit. Le salaire pour M1, M2 et M5 ; le Compa-Ratio pour M3, M4 et M6. Les deux ne se comparent pas entre eux.

Lecture : dans une colonne (un modèle), la ligne qui a la plus petite MAE est la meilleure façon de former les cohortes pour ce modèle. Si la dernière ligne, « Sans cohorte », a la plus petite MAE, apprendre par cohorte ne fait pas mieux que le modèle général.

Ce qui joue contre les cohortes précises : chaque modèle n'a que les salariés de sa cohorte pour apprendre. Plus la cohorte est précise, moins il en a. Un arbre ou un gradient boosting ont besoin de beaucoup de monde ; dans une petite cohorte, ils apprennent mal.

---
## 5. Le classement, du meilleur au pire

Pour chaque façon de former les cohortes, on garde son meilleur modèle : celui qui a la plus petite MAE. Puis on classe les façons de faire, de la plus petite MAE à la plus grande.

Le repère, « Sans cohorte », est classé avec les autres. Les façons de faire placées au-dessus de lui font mieux que le modèle général ; celles placées en dessous font moins bien.

In [ ]:
# 1. Pour chaque façon de former les cohortes (repère compris) : son meilleur modèle, celui qui a la plus petite MAE
classement = resultats.loc[resultats.groupby('Cohorte')['MAE (MAD)'].idxmin()]
# 2. Du meilleur au pire : de la plus petite MAE à la plus grande
classement = classement.sort_values('MAE (MAD)').reset_index(drop=True)
classement.index = classement.index + 1   # le rang : 1 pour le meilleur
classement.insert(2, 'Ce que fait le modèle', classement['Modèle'].map({nom: quoi for nom, (quoi, cible, modele) in MODELES.items()}))
display(classement.round(2))